In [12]:
import pandas as pd
import os

# Load data from new file structure
# Current script location: folder1/
# Data files location: folder1/Data/{fund_name}/{file_name}

df_aware = pd.read_csv('Data/aware/aware_cleaned.csv')
df_cbus = pd.read_csv('Data/cbus/cbus_cleaned.csv')
df_ngs = pd.read_csv('Data/ngs/ngs_cleaned.csv')
df_rest = pd.read_csv('Data/rest/rest_cleaned.csv')

# Weight calculations for all funds
funds_data = {
    'AWARE': {
        'total_weight': df_aware['Weighting (%)'].sum(),
        'sub_weight': df_aware[df_aware['Name/Kind of Investment Item'] == 'Sub Total']['Weighting (%)'].sum(),
        'no_sub_weight': df_aware[df_aware['Name/Kind of Investment Item'] != 'Sub Total']['Weighting (%)'].sum()
    },
    'CBUS': {
        'total_weight': df_cbus['Weighting (%)'].sum(),
        'sub_weight': df_cbus[df_cbus['Name/Kind of Investment Item'] == 'Sub Total']['Weighting (%)'].sum(),
        'no_sub_weight': df_cbus[df_cbus['Name/Kind of Investment Item'] != 'Sub Total']['Weighting (%)'].sum()
    },
    'NGS': {
        'total_weight': df_ngs['Weighting (%)'].sum(),
        'sub_weight': df_ngs[df_ngs['Name/Kind of Investment Item'] == 'Sub Total']['Weighting (%)'].sum(),
        'no_sub_weight': df_ngs[df_ngs['Name/Kind of Investment Item'] != 'Sub Total']['Weighting (%)'].sum()
    },
    'REST': {
        'total_weight': df_rest['Weighting (%)'].sum(),
        'sub_weight': df_rest[df_rest['Name/Kind of Investment Item'] == 'Sub Total']['Weighting (%)'].sum(),
        'no_sub_weight': df_rest[df_rest['Name/Kind of Investment Item'] != 'Sub Total']['Weighting (%)'].sum()
    }
}

# Calculate total_check and print results for each fund
print("WEIGHT VALIDATION RESULTS")
print("=" * 50)

for fund_name, data in funds_data.items():
    total_check = data['sub_weight'] + data['no_sub_weight']
    is_valid = abs(data['total_weight'] - total_check) < 0.01  # Allow small floating point differences
    
    print(f"\n{fund_name}:")
    print(f"  Total Weight: {data['total_weight']:.2f}")
    print(f"  Sub Total Weight: {data['sub_weight']:.2f}")
    print(f"  Non-Sub Weight: {data['no_sub_weight']:.2f}")
    print(f"  Total Check: {total_check:.2f}")
    print(f"  Validation: {'✓ PASS' if is_valid else '✗ FAIL'}")
    
    if not is_valid:
        difference = data['total_weight'] - total_check
        print(f"  Difference: {difference:.2f}")

print("\n" + "=" * 50)
print("SUMMARY:")
all_valid = all(abs(data['total_weight'] - (data['sub_weight'] + data['no_sub_weight'])) < 0.01 
                for data in funds_data.values())
print(f"All funds validated: {'✓ YES' if all_valid else '✗ NO'}")


for file in files:
    exists = os.path.exists(file)
    print(f"  {file}: {'✓ Found' if exists else '✗ Not Found'}")

WEIGHT VALIDATION RESULTS

AWARE:
  Total Weight: 1.88
  Sub Total Weight: 0.00
  Non-Sub Weight: 1.88
  Total Check: 1.88
  Validation: ✓ PASS

CBUS:
  Total Weight: 1.94
  Sub Total Weight: 0.00
  Non-Sub Weight: 1.94
  Total Check: 1.94
  Validation: ✓ PASS

NGS:
  Total Weight: 1.96
  Sub Total Weight: 0.00
  Non-Sub Weight: 1.96
  Total Check: 1.96
  Validation: ✓ PASS

REST:
  Total Weight: 3.01
  Sub Total Weight: 0.00
  Non-Sub Weight: 3.01
  Total Check: 3.01
  Validation: ✓ PASS

SUMMARY:
All funds validated: ✓ YES
  Data/aware/aware_cleaned.csv: ✓ Found
  Data/cbus/cbus_cleaned.csv: ✓ Found
  Data/ngs/ngs_cleaned.csv: ✓ Found
  Data/rest/rest_cleaned.csv: ✓ Found


In [13]:
import pandas as pd

# Load all datasets
df_aware = pd.read_csv('Data/aware/aware_cleaned.csv')
df_cbus = pd.read_csv('Data/cbus/cbus_cleaned.csv')
df_ngs = pd.read_csv('Data/ngs/ngs_cleaned.csv')
df_rest = pd.read_csv('Data/rest/rest_cleaned.csv')

# Store them in a dict
funds = {
    'AWARE': df_aware,
    'CBUS': df_cbus,
    'NGS': df_ngs,
    'REST': df_rest
}

# Final results container
results = []

for fund_name, df in funds.items():
    # Lowercase relevant columns
    df['Asset Class Name'] = df['Asset Class Name'].str.lower()
    df['Name/Kind of Investment Item'] = df['Name/Kind of Investment Item'].str.lower()

    # Remove rows where the investment item is a subtotal
    df = df[~df['Name/Kind of Investment Item'].str.contains("sub total", na=False)]

    # Determine the target equity labels
    if fund_name == 'CBUS':
        target_equities = ['listed equities', 'unlisted equities']
    else:
        target_equities = ['listed equity', 'unlisted equity']

    # Filter for only listed/unlisted equity
    filtered = df[df['Asset Class Name'].isin(target_equities)]

    # Sum the weighting
    equity_weight = filtered['Weighting (%)'].sum()

    # Append result
    results.append({
        'Fund Name': fund_name,
        'Equity Weighting': equity_weight
    })

# Create summary DataFrame
summary_df = pd.DataFrame(results)

# Display result
print(summary_df)


  Fund Name  Equity Weighting
0     AWARE            0.5828
1      CBUS            0.5671
2       NGS            0.5615
3      REST            0.3808
